# Natural Language Processing
## Assignment Number: 1
## Student Name     : (John) Paul Nagle
## Student ID       : R00065426

In [1]:
# Run this cell once to install all required packages
#!pip install transformers datasets peft accelerate torch sentencepiece -q

## Imports

In [ ]:
import warnings
import re
import unicodedata


warnings.filterwarnings("ignore")

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering,
    pipeline,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset

In [19]:
class QuestionAnswerer:
    """Question answering system with text normalization."""
    
    def __init__(self, model_name: str = "distilbert-base-cased-distilled-squad"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        
        # Normalization config
        self.max_input_length = 512
        self.min_input_length = 5
    
    def normalize_input(self, text: str) -> str:
        """Multi-stage input normalization."""
        # Unicode normalization
        text = unicodedata.normalize('NFC', text)
        
        # Remove/replace problematic characters
        text = text.replace('\r\n', '\n').replace('\r', '\n')
        text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]', '', text)
        
        # Normalize whitespace
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = text.strip()
        
        return text
    
    def normalize_output(self, text: str) -> str:
        """Post-process extracted answer."""
        # Fix spacing around punctuation
        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        text = re.sub(r'([.,!?;:])([^\s])', r'\1 \2', text)
        
        return text.strip()
    
    def answer_question(self, question: str, context: str) -> dict:
        """Answer question based on context."""
        
        # 1. Normalize inputs
        question = self.normalize_input(question)
        context = self.normalize_input(context)
        
        # 2. Tokenize question + context together
        inputs = self.tokenizer(
            question,
            context,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_input_length,
            padding=True
        )
        
        # 3. Get model predictions
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # 4. Extract answer span from logits
        answer_start = torch.argmax(outputs.start_logits)
        answer_end = torch.argmax(outputs.end_logits) + 1
        
        # 5. Decode the answer
        answer = self.tokenizer.decode(
            inputs["input_ids"][0][answer_start:answer_end],
            skip_special_tokens=True
        )
        
        # 6. Calculate confidence score
        start_score = float(torch.max(outputs.start_logits))
        end_score = float(torch.max(outputs.end_logits))
        confidence = (start_score + end_score) / 2
        
        return {
            "answer": self.normalize_output(answer),
            "confidence": confidence,
            "start_position": int(answer_start),
            "end_position": int(answer_end)
        }


In [20]:
# Initialize the QA system
qa = QuestionAnswerer()

# Example 1: Simple factual question
context1 = """
Tokyo is the capital of Japan and one of the most populous cities in the world.
It is located on the eastern coast of Honshu island and serves as the political,
economic, and cultural center of Japan.
"""

result1 = qa.answer_question("What is the capital of Japan?", context1)
print(f"Question: What is the capital of Japan?")
print(f"Answer: {result1['answer']}")
print(f"Confidence: {result1['confidence']:.2f}\n")
# Output: Answer: Tokyo

# Example 2: Location question
context2 = """
The Eiffel Tower is located in Paris, France. It was built in 1889 and
stands 330 meters tall. It is one of the most visited monuments in the world.
"""

result2 = qa.answer_question("Where is the Eiffel Tower?", context2)
print(f"Question: Where is the Eiffel Tower?")
print(f"Answer: {result2['answer']}")
print(f"Confidence: {result2['confidence']:.2f}\n")
# Output: Answer: Paris, France

# Example 3: Numerical question
context3 = """
Machine learning is a subset of artificial intelligence that was developed
in the 1950s. It enables computers to learn from data without being explicitly
programmed.
"""

result3 = qa.answer_question("When was machine learning developed?", context3)
print(f"Question: When was machine learning developed?")
print(f"Answer: {result3['answer']}")
print(f"Confidence: {result3['confidence']:.2f}\n")
# Output: Answer: 1950s

# Example 4: Handling messy input (demonstrates normalization)
messy_context = "Deep\n\n\nlearning   models  are   powerful"
messy_question = "What   are    powerful?"

result4 = qa.answer_question(messy_question, messy_context)
print(f"Question: {messy_question}")
print(f"Answer: {result4['answer']}")
# Output: Answer: Deep learning models




Loading weights: 100%|██████████| 102/102 [00:00<00:00, 18046.87it/s]


Question: What is the capital of Japan?
Answer: Tokyo
Confidence: 10.25

Question: Where is the Eiffel Tower?
Answer: Paris, France
Confidence: 10.57

Question: When was machine learning developed?
Answer: 1950s
Confidence: 10.96

Question: What   are    powerful?
Answer: Deep learning models
